# 02 — Data Cleaning & Wrangling

**Stage 2 of the Tier A pipeline** · input `data/raw/housing.csv` → output `data/processed/boston_clean.parquet`

> **This dataset arrives clean.** Stage 1 confirmed 0 nulls, 0 duplicates, and 14/14 schema checks
> passing. So this notebook is not repair work — it is where three **structural decisions** get made
> and documented, each of which silently changes every downstream number if got wrong:
>
> 1. **`MEDV` is top-coded at 50.0** — 16 tracts are censored, not observed maxima.
> 2. **`RAD` is an index, not a distance** — it jumps 1–8 then straight to 24.
> 3. **`CRIM`'s long tail is real** — high-crime tracts are signal, not error.
>
> STRUCTURE.md warns that cleaning "typically consumes 50–70% of project time." When it doesn't, the
> honest move is to say so and spend the time on the decisions that *do* matter, rather than
> performing cleaning theatre on data that needs none.

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROJECT = Path.cwd().parent
RAW = PROJECT / "data" / "raw" / "housing.csv"
PROC_DIR = PROJECT / "data" / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

df = pd.read_csv(RAW)
print(f"loaded {df.shape[0]} rows x {df.shape[1]} columns from data/raw/housing.csv")

# Every action taken in this notebook appends here. Nothing changes silently.
cleaning_log: list[dict] = []

def log(action: str, column: str, rows_affected: int, why: str) -> None:
    cleaning_log.append({
        "action": action, "column": column,
        "rows_affected": rows_affected, "rationale": why,
    })
    print(f"[{action}] {column}: {rows_affected} rows -- {why}")

loaded 506 rows x 14 columns from data/raw/housing.csv


## 2.1 Missingness and duplicates — verified, not assumed

Re-derived independently of the data notes. A cleaning strategy is only defensible if the problem it
solves is measured first.

In [2]:
miss = pd.DataFrame({
    "n_missing": df.isna().sum(),
    "pct_missing": (df.isna().mean() * 100).round(2),
})
n_missing_total = int(miss["n_missing"].sum())
n_dupes = int(df.duplicated().sum())

print(f"total missing cells : {n_missing_total}")
print(f"duplicate rows      : {n_dupes}")

if n_missing_total == 0:
    log("no-op", "ALL", 0,
        "0 missing cells across all 14 columns -- no imputation strategy required. "
        "Missingness mechanism (MCAR/MAR/MNAR) is moot where nothing is missing.")
if n_dupes == 0:
    log("no-op", "ALL", 0,
        "0 exact duplicate rows. Note: tracts are distinct geographic units, so genuine "
        "duplicates would indicate an ingestion fault rather than legitimate repeat observations.")

miss.T

total missing cells : 0
duplicate rows      : 0
[no-op] ALL: 0 rows -- 0 missing cells across all 14 columns -- no imputation strategy required. Missingness mechanism (MCAR/MAR/MNAR) is moot where nothing is missing.
[no-op] ALL: 0 rows -- 0 exact duplicate rows. Note: tracts are distinct geographic units, so genuine duplicates would indicate an ingestion fault rather than legitimate repeat observations.


,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
n_missing,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
pct_missing,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


> **So What:** no rows are dropped or imputed, so the full 506-tract sample survives into analysis.
> There is no missing-data caveat to attach to any downstream estimate.

## 2.2 Dtypes — `RAD` is an accessibility *index*, and treating it as continuous is a silent error

`RAD` runs 1–8 and then leaps to 24 with nothing in between. Fitting a linear term to it asserts that
the step from index 23 to 24 means the same thing as 1 to 2 — a statement the data cannot support.
It becomes an **ordered categorical**.

In [3]:
print("RAD value counts (note the gap between 8 and 24):")
print(df["RAD"].value_counts().sort_index().to_string())

gap = sorted(df["RAD"].unique())
print(f"\nobserved levels: {gap}")
print(f"largest gap between consecutive levels: {max(np.diff(gap))}")

RAD value counts (note the gap between 8 and 24):
RAD
1      20
2      24
3      38
4     110
5     115
6      26
7      17
8      24
24    132

observed levels: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(24)]
largest gap between consecutive levels: 16


In [4]:
before = df.dtypes.astype(str).copy()

# CHAS: binary dummy -> int8. Semantically a flag, not a measurement.
df["CHAS"] = df["CHAS"].astype("int8")
log("cast", "CHAS", len(df), "binary river-adjacency dummy -> int8; it is a flag, not a quantity")

# RAD: ordered categorical. Prevents any model from interpolating across the 8->24 gap.
df["RAD"] = pd.Categorical(df["RAD"].astype(int), categories=sorted(df["RAD"].astype(int).unique()), ordered=True)
log("cast", "RAD", len(df),
    "accessibility INDEX with a 8->24 discontinuity -> ordered categorical; a linear term "
    "would falsely assert equal spacing between adjacent indices")

# Everything else is a genuine continuous measurement.
for c in ["CRIM", "ZN", "INDUS", "NOX", "RM", "AGE", "DIS", "TAX", "PTRATIO", "B", "LSTAT", "MEDV"]:
    df[c] = df[c].astype("float64")
log("cast", "12 continuous columns", len(df), "explicit float64 to prevent silent integer coercion")

pd.DataFrame({"before": before, "after": df.dtypes.astype(str)})

[cast] CHAS: 506 rows -- binary river-adjacency dummy -> int8; it is a flag, not a quantity
[cast] RAD: 506 rows -- accessibility INDEX with a 8->24 discontinuity -> ordered categorical; a linear term would falsely assert equal spacing between adjacent indices
[cast] 12 continuous columns: 506 rows -- explicit float64 to prevent silent integer coercion


,before,after
CRIM,float64,float64
ZN,float64,float64
INDUS,float64,float64
CHAS,int64,int8
NOX,float64,float64
RM,float64,float64
AGE,float64,float64
DIS,float64,float64
RAD,int64,category
TAX,float64,float64


## 2.3 The censoring decision — 16 tracts are top-coded, and this biases every high-end estimate

`MEDV == 50.0` exactly, for 16 tracts. In a continuous median-value variable, an exact repeated
maximum is the signature of **top-coding**: the survey recorded "$50,000 or more" and the true values
are unknown but *at least* 50.

This is the single most consequential decision in the notebook. Ignoring it means every model is
trained to predict a ceiling that does not exist in reality, and every high-end coefficient is
attenuated toward zero.

In [5]:
censored = df["MEDV"] == 50.0
n_cens = int(censored.sum())

# Is 50.0 a genuine mode, or an artificial pile-up? Compare with the next-highest values.
top = df["MEDV"].value_counts().sort_index(ascending=False).head(8)
print("highest MEDV values and their frequencies:")
print(top.to_string())
print(f"\n{n_cens} tracts sit at exactly 50.0 -- {n_cens / len(df):.1%} of the sample")
print(f"next distinct value below: {sorted(df.MEDV.unique())[-2]}")

df["is_MEDV_censored"] = censored.astype("int8")
log("flag", "is_MEDV_censored", n_cens,
    "MEDV top-coded at 50.0 (survey ceiling). Flag created; rows RETAINED for the primary "
    "analysis. Sensitivity re-run on n=490 is scheduled in 04_analysis.")

highest MEDV values and their frequencies:
MEDV
50.0    16
48.8     1
48.5     1
48.3     1
46.7     1
46.0     1
45.4     1
44.8     1

16 tracts sit at exactly 50.0 -- 3.2% of the sample
next distinct value below: 48.8
[flag] is_MEDV_censored: 16 rows -- MEDV top-coded at 50.0 (survey ceiling). Flag created; rows RETAINED for the primary analysis. Sensitivity re-run on n=490 is scheduled in 04_analysis.


In [6]:
# How different are the censored tracts? If they are ordinary, dropping them would be low-risk;
# if they are systematically distinctive, dropping them would bias the sample.
profile = (
    df.groupby(censored.map({True: "censored (MEDV=50)", False: "observed (MEDV<50)"}))[
        ["RM", "LSTAT", "CRIM", "PTRATIO", "NOX"]
    ]
    .mean()
    .round(2)
)
profile["n"] = censored.map({True: "censored (MEDV=50)", False: "observed (MEDV<50)"}).value_counts()
profile

,RM,LSTAT,CRIM,PTRATIO,NOX,n
MEDV,,,,,,
censored (MEDV=50),7.48,4.36,2.70,16.48,0.57,16
observed (MEDV<50),6.25,12.92,3.64,18.52,0.55,490


> **So What:** the censored tracts are not a random 3% — they have markedly more rooms and a far
> lower share of low-status population than the rest of the sample. Dropping them would delete the
> top of the value distribution and flatten the very relationships the analysis exists to measure.
>
> **Implication:** retain all 506 rows for the primary analysis, carry the flag, and report every
> high-end estimate as **attenuated** — the true `RM` effect is *at least* what we measure, not
> exactly it. A 490-row sensitivity run in `04_analysis` quantifies how much this matters.

## 2.4 Outliers — `CRIM`'s long tail is real and stays

`CRIM` reaches 89 per capita against a median near 0.26. STRUCTURE.md is explicit: *never remove
outliers without a documented reason*. Here the documented reason runs the other way — these are
genuine high-crime tracts, and they are precisely the neighbourhoods a planning budget would target.
Deleting them would delete the problem.

In [7]:
def iqr_outliers(s: pd.Series) -> int:
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum())

cont = ["CRIM", "ZN", "INDUS", "NOX", "RM", "AGE", "DIS", "TAX", "PTRATIO", "B", "LSTAT", "MEDV"]
out = pd.DataFrame({
    "n_iqr_outliers": {c: iqr_outliers(df[c]) for c in cont},
    "skew": df[cont].skew().round(2),
    "min": df[cont].min().round(2),
    "median": df[cont].median().round(2),
    "max": df[cont].max().round(2),
}).sort_values("n_iqr_outliers", ascending=False)

log("retain", "CRIM / ZN / B / others", 0,
    "IQR flags many points, but all are substantively plausible tract values, not recording "
    "errors. Retained in full; right-skew is handled by log transforms at feature-engineering "
    "time (Stage 4), never by deletion.")
out

[retain] CRIM / ZN / B / others: 0 rows -- IQR flags many points, but all are substantively plausible tract values, not recording errors. Retained in full; right-skew is handled by log transforms at feature-engineering time (Stage 4), never by deletion.


,n_iqr_outliers,skew,min,median,max
B,77,-2.89,0.32,391.44,396.90
ZN,68,2.23,0.00,0.00,100.00
CRIM,66,5.22,0.01,0.26,88.98
MEDV,40,1.11,5.00,21.20,50.00
RM,30,0.40,3.56,6.21,8.78
PTRATIO,15,-0.80,12.60,19.05,22.00
LSTAT,7,0.91,1.73,11.36,37.97
DIS,5,1.01,1.13,3.21,12.13
NOX,0,0.73,0.38,0.54,0.87
INDUS,0,0.30,0.46,9.69,27.74


> **So What:** zero rows are removed for being extreme. Skew is a *modelling* problem, solved with
> transformations in Stage 4, not a *data quality* problem solved with a delete key.

## 2.5 Business-rule validation

Assertions that must hold for the data to be internally coherent. These are hard gates — the notebook
fails loudly rather than writing a corrupt artifact.

In [8]:
rules = [
    ("MEDV strictly positive",            (df.MEDV > 0).all()),
    ("CHAS is binary",                    set(df.CHAS.unique()) <= {0, 1}),
    ("proportions within 0-100",          all(df[c].between(0, 100).all() for c in ["ZN", "INDUS", "AGE", "LSTAT"])),
    ("NOX within documented 0.30-0.90",   df.NOX.between(0.30, 0.90).all()),
    ("RM physically plausible (1-15)",    df.RM.between(1, 15).all()),
    ("DIS strictly positive",             (df.DIS > 0).all()),
    ("TAX strictly positive",             (df.TAX > 0).all()),
    ("PTRATIO strictly positive",         (df.PTRATIO > 0).all()),
    ("censoring flag matches MEDV==50",   ((df.is_MEDV_censored == 1) == (df.MEDV == 50)).all()),
    ("row count preserved at 506",        len(df) == 506),
    ("no nulls introduced by cleaning",   df.isna().sum().sum() == 0),
]
validation = pd.DataFrame(rules, columns=["rule", "passes"])
validation["status"] = np.where(validation.passes, "PASS", "FAIL")

assert validation.passes.all(), f"business rules violated: {validation.loc[~validation.passes, 'rule'].tolist()}"
print(f"all {len(validation)} business rules PASS")
validation[["rule", "status"]]

all 11 business rules PASS


,rule,status
0,MEDV strictly positive,PASS
1,CHAS is binary,PASS
2,proportions within 0-100,PASS
3,NOX within documented 0.30-0.90,PASS
4,RM physically plausible (1-15),PASS
5,DIS strictly positive,PASS
6,TAX strictly positive,PASS
7,PTRATIO strictly positive,PASS
8,censoring flag matches MEDV==50,PASS
9,row count preserved at 506,PASS


## 2.6 Write the processed dataset and the cleaning log

> **Platform quirk found here, not later.** `pandas 3.0.3` + `pyarrow 25.0.0` silently discard the
> ordered-categorical dtype on a parquet round-trip — `RAD` comes back as plain `int64`, quietly
> undoing the §2.2 decision. Parquet stores the values, not the pandas dtype contract.
>
> The fix is to make the contract an explicit, versioned artifact (`_dtype_contract.json`) that
> downstream notebooks re-apply on load. The assertion below is what caught this; it stays in place
> so the failure can never become silent again.

In [9]:
import json

out_path = PROC_DIR / "boston_clean.parquet"
df.to_parquet(out_path, index=False)

# The dtype contract travels beside the data because parquet will not carry it.
DTYPE_CONTRACT = {
    "ordered_categorical": {"RAD": [int(c) for c in df["RAD"].cat.categories]},
    "int8": ["CHAS", "is_MEDV_censored"],
    "float64": ["CRIM", "ZN", "INDUS", "NOX", "RM", "AGE", "DIS",
                "TAX", "PTRATIO", "B", "LSTAT", "MEDV"],
    "note": "pandas/pyarrow drop categorical dtypes through parquet. Downstream notebooks MUST "
            "re-apply this contract on load -- see apply_dtype_contract().",
}
(PROC_DIR / "_dtype_contract.json").write_text(json.dumps(DTYPE_CONTRACT, indent=2))


def apply_dtype_contract(frame: pd.DataFrame, contract: dict) -> pd.DataFrame:
    """Restore the dtypes parquet cannot preserve. Re-defined in every downstream notebook."""
    frame = frame.copy()
    for col, cats in contract["ordered_categorical"].items():
        frame[col] = pd.Categorical(frame[col].astype(int), categories=cats, ordered=True)
    for col in contract["int8"]:
        frame[col] = frame[col].astype("int8")
    for col in contract["float64"]:
        frame[col] = frame[col].astype("float64")
    return frame


log_df = pd.DataFrame(cleaning_log)
log_df.to_csv(PROC_DIR / "_cleaning_log.csv", index=False)
validation.to_csv(PROC_DIR / "_business_rules.csv", index=False)

# Round-trip WITH the contract re-applied -- this is the real reproducibility test.
check = apply_dtype_contract(pd.read_parquet(out_path), DTYPE_CONTRACT)
assert check.shape == df.shape, "parquet round-trip changed the shape"
assert str(check["RAD"].dtype) == "category" and check["RAD"].cat.ordered, "RAD contract not restored"
assert (check["RAD"].astype(int) == df["RAD"].astype(int)).all(), "RAD values changed"
assert np.allclose(check[DTYPE_CONTRACT["float64"]], df[DTYPE_CONTRACT["float64"]]), "values changed"

print(f"wrote data/processed/boston_clean.parquet  ({check.shape[0]} x {check.shape[1]})")
print("round-trip verified: values identical AND dtype contract restored")
print("")
print(f"cleaning log: {len(log_df)} actions recorded, 0 rows dropped, 0 values imputed")
log_df

wrote data/processed/boston_clean.parquet  (506 x 15)
round-trip verified: values identical AND dtype contract restored

cleaning log: 7 actions recorded, 0 rows dropped, 0 values imputed


,action,column,rows_affected,rationale
0,no-op,ALL,0,0 missing cells across all 14 columns -- no im...
1,no-op,ALL,0,0 exact duplicate rows. Note: tracts are disti...
2,cast,CHAS,506,binary river-adjacency dummy -> int8; it is a ...
3,cast,RAD,506,accessibility INDEX with a 8->24 discontinuity...
4,cast,12 continuous columns,506,explicit float64 to prevent silent integer coe...
5,flag,is_MEDV_censored,16,MEDV top-coded at 50.0 (survey ceiling). Flag ...
6,retain,CRIM / ZN / B / others,0,"IQR flags many points, but all are substantive..."


---

## Stage 2 — Gate Checklist

- [x] **Missing value strategy documented per column** — 0 missing cells measured across all 14 columns; no imputation required, and the absence is recorded in the cleaning log rather than assumed
- [x] **Duplicates assessed and handled with justification** — 0 exact duplicates; noted that tracts are distinct geographic units, so any duplicate would signal an ingestion fault
- [x] **All columns have correct dtypes** — `CHAS`→int8 flag, `RAD`→**ordered categorical** (blocks false interpolation across the 8→24 gap), 12 measurements→float64
- [x] **Business-rule validations pass** — 11/11 assertions, enforced with `assert` so a violation fails the notebook
- [x] **Cleaning log exists** — `_cleaning_log.csv`: every action with column, rows affected, and rationale
- [x] **Cleaned data saved to `data/processed/`** — `boston_clean.parquet` + `_dtype_contract.json`; round-trip verified for both values *and* restored dtypes

**Gate status: PASS → proceed to `03_eda.ipynb`.**

### Decisions carried forward
| Decision | Choice | Downstream consequence |
|---|---|---|
| 16 top-coded `MEDV=50` tracts | **Retained + flagged**, not dropped | High-end estimates are attenuated; n=490 sensitivity run required in Stage 6 |
| `RAD` | **Ordered categorical** | One-hot encoded for models; cannot enter as a linear term |
| `CRIM` long tail | **Retained in full** | Right-skew handled by `log1p` in Stage 4, not by row deletion |
| `B` | Retained in the file, **barred from models** | Fairness audit in Stage 6 uses it as an auditing target only |

### Changelog
| Pass | Date | Change |
|---|---|---|
| 1 | 2026-07-20 | Initial cleaning. 0 rows dropped, 0 values imputed, 0 values edited. Three structural dtype/censoring decisions recorded above. |
| 2 | 2026-07-20 | Round-trip assertion caught pandas/pyarrow silently dropping `RAD`'s ordered-categorical dtype. Added `_dtype_contract.json` + `apply_dtype_contract()` so the Stage 2 decision survives into Stages 3–6. |